In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

DataFrame[]

In [0]:
# ---------------------------
# 1️. Read Bronze Delta Tables
# ---------------------------
transactions = spark.table("bronze.transactions")
cards = spark.table("bronze.cards")
users = spark.table("bronze.users")
mcc = spark.table("bronze.mcc")
fraud = spark.table("bronze.train_fraud")

display(transactions.limit(5))
display(cards.limit(5))
display(users.limit(5))
display(mcc.limit(5))
display(fraud.limit(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00Z,1556,2972,$-77.00,Swipe Transaction,59935,Beulah,ND,58523,5499,null
7475328,2010-01-01T00:02:00Z,561,4575,$14.57,Swipe Transaction,67570,Bettendorf,IA,52722,5311,null
7475329,2010-01-01T00:02:00Z,1129,102,$80.00,Swipe Transaction,27092,Vista,CA,92084,4829,null
7475331,2010-01-01T00:05:00Z,430,2860,$200.00,Swipe Transaction,27092,Crown Point,IN,46307,4829,null
7475332,2010-01-01T00:06:00Z,848,3915,$46.41,Swipe Transaction,13051,Harwood,MD,20776,5813,null


id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,2022-12-01,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,2020-12-01,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,2024-02-01,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,2024-08-01,693,NO,1,$12400,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-01,75,YES,1,$28,09/2008,2009,No


id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,$29278,$59696,$127613,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,$37891,$77254,$191349,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,$22681,$33483,$196,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,$163145,$249925,$202328,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,$53797,$109687,$183855,675,1


mcc,description
1711,"Heating, Plumbing, Air Conditioning Contractors"
3000,Steelworks
3001,Steel Products Manufacturing
3005,Miscellaneous Metal Fabrication
3006,Miscellaneous Fabricated Metal Products


transaction_id,fraud_label
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No


In [0]:
from pyspark.sql.functions import col, regexp_replace

transactions_clean = transactions.select(
    col("id").alias("transaction_id"),
    col("date").alias("transaction_date"),
    col("client_id"),
    col("card_id"),

    # Remove $ and convert to float
    regexp_replace(col("amount"), "[$]", "").cast("float").alias("amount"),

    col("use_chip"),
    col("merchant_id"),
    col("merchant_city"),
    col("merchant_state"),
    col("zip"),
    col("mcc"),
    col("errors")
)

display(transactions_clean.limit(5))

transaction_id,transaction_date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
7475327,2010-01-01T00:01:00Z,1556,2972,-77.0,Swipe Transaction,59935,Beulah,ND,58523,5499,null
7475328,2010-01-01T00:02:00Z,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,5311,null
7475329,2010-01-01T00:02:00Z,1129,102,80.0,Swipe Transaction,27092,Vista,CA,92084,4829,null
7475331,2010-01-01T00:05:00Z,430,2860,200.0,Swipe Transaction,27092,Crown Point,IN,46307,4829,null
7475332,2010-01-01T00:06:00Z,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,5813,null


In [0]:
# ---------------------------
# 2️. Enhance Transactions with Fraud Labels and Merchant Category
# ---------------------------
from pyspark.sql.functions import col

silver_transactions = transactions_clean \
    .join(fraud, on="transaction_id", how="left") \
    .join(mcc, on="mcc", how="left") \
    .withColumnRenamed("description", "merchant_category")

# Preview enhanced transactions
display(silver_transactions.limit(10))

mcc,transaction_id,transaction_date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,fraud_label,merchant_category
4784,7475335,2010-01-01T00:14:00Z,1684,2140,26.46,Online Transaction,39021,ONLINE,null,null,null,No,Tolls and Bridge Fees
4829,15655095,2015-02-09T19:07:00Z,114,3070,80.0,Chip Transaction,27092,Baldwin Park,CA,91706,Insufficient Balance,No,Money Transfer
5942,7475333,2010-01-01T00:07:00Z,1807,165,4.81,Swipe Transaction,20519,Bronx,NY,10464,null,No,Book Stores
4121,15655099,2015-02-09T19:08:00Z,526,5917,22.17,Chip Transaction,47976,Elkins,WV,26241,null,null,Taxicabs and Limousines
5311,7475328,2010-01-01T00:02:00Z,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,null,No,Department Stores
5499,7475327,2010-01-01T00:01:00Z,1556,2972,-77.0,Swipe Transaction,59935,Beulah,ND,58523,null,No,Miscellaneous Food Stores
5813,7475332,2010-01-01T00:06:00Z,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,null,No,Drinking Places (Alcoholic Beverages)
5499,15655103,2015-02-09T19:10:00Z,316,5857,78.56,Chip Transaction,59935,Houston,TX,77049,null,No,Miscellaneous Food Stores
4829,7475329,2010-01-01T00:02:00Z,1129,102,80.0,Swipe Transaction,27092,Vista,CA,92084,null,No,Money Transfer
5310,15655096,2015-02-09T19:07:00Z,1494,3579,1.31,Chip Transaction,96507,Fort Lauderdale,FL,33311,null,No,Discount Stores


In [0]:
# ---------------------------
# 3️. Save Silver Transactions Table
# ---------------------------
silver_transactions.write.format("delta").mode("overwrite").saveAsTable("silver.transactions")

In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date, when

cards_clean = cards.select(
    col("id").alias("card_id"),
    col("client_id"),
    col("card_brand"),
    col("card_type"),
    col("card_number"),

    # Convert to date
    to_date(col("expires")).alias("expires"),

    col("cvv"),

    # Convert YES/NO to boolean
    when(col("has_chip") == "YES", True).otherwise(False).alias("has_chip"),

    col("num_cards_issued"),

    # Remove $ and convert to float
    regexp_replace(col("credit_limit"), "[$]", "").cast("float").alias("credit_limit"),

    # Convert to date
    to_date(col("acct_open_date"), "MM/yyyy").alias("acct_open_date"),

    col("year_pin_last_changed"),

    # Convert Yes/No to boolean
    when(col("card_on_dark_web") == "Yes", True).otherwise(False).alias("card_on_dark_web")
)

display(cards_clean.limit(5))

card_id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,2022-12-01,623,true,2,24295.0,2002-09-01,2008,false
2731,825,Visa,Debit,4956965974959986,2020-12-01,393,true,2,21968.0,2014-04-01,2014,false
3701,825,Visa,Debit,4582313478255491,2024-02-01,719,true,2,46414.0,2003-07-01,2004,false
42,825,Visa,Credit,4879494103069057,2024-08-01,693,false,1,12400.0,2003-01-01,2012,false
4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-01,75,true,1,28.0,2008-09-01,2009,false


In [0]:
cards_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.cards")

In [0]:
from pyspark.sql.functions import col, regexp_replace

users_clean = users.select(
    col("id").alias("user_id"),
    col("current_age"),
    col("retirement_age"),
    col("birth_year"),
    col("birth_month"),
    col("gender"),
    col("address"),
    col("latitude"),
    col("longitude"),

    # Remove $ and convert to float
    regexp_replace(col("per_capita_income"), "[$]", "").cast("float").alias("per_capita_income"),
    regexp_replace(col("yearly_income"), "[$]", "").cast("float").alias("yearly_income"),
    regexp_replace(col("total_debt"), "[$]", "").cast("float").alias("total_debt"),

    col("credit_score"),
    col("num_credit_cards")
)

display(users_clean.limit(5))

user_id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [0]:
users_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.users")